In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# set the font size
plt.rcParams.update({'font.size': 7})
# set Helvetica globally
plt.rcParams['font.family'] = 'Helvetica'

plot_folder = "plots"
os.makedirs(plot_folder, exist_ok=True)

prefix = "lib1_ActD"

In [ ]:
# This should be added from the data repo
if prefix == "lib2_ActD":
    count_df = pd.read_csv('../count_data/overall_lib2_processing/counts_out/K562-ActD-lib2-counts-processed.csv', index_col=0)
    count_df = count_df.fillna(0)
    count_df.columns = count_df.columns.str.replace("K562_3UTR_", "")
elif prefix == "lib1_ActD":
    count_df = pd.read_csv('../count_data/overall_lib1_processing/counts_out/K562-ActD-lib1-counts-processed.csv', index_col=0)
    count_df = count_df.fillna(0)
    count_df.columns = count_df.columns.str.replace("K562_3UTR_", "")
else:
    raise ValueError(f"Prefix {prefix} not recognized")

In [ ]:
time_points = [0.08, 0.5, 1, 2, 4, 8, 12, 24] # first time point isn't 0 but instead ~5 min to account for the delay in the start of the experiment

In [ ]:
if prefix == "lib2_ActD":
    mirna_index = list(count_df[count_df.index.str.lower().str.contains("mirna")].index)
    lib2_ctrl_index = list(count_df[count_df.index.str.contains("0_lib2_controls")].index)
    ctrl_index = list(count_df[count_df.index.str.contains("P96")].index)
    count_df= count_df[count_df.index.isin(mirna_index + lib2_ctrl_index + ctrl_index)]
    count_df = count_df.dropna()
    
elif prefix == "lib1_ActD":
    mirna_index = list(count_df[count_df.index.str.lower().str.contains("mirna")].index)
    ctrl_index = list(count_df[count_df.index.str.contains("P88")].index)
    count_df= count_df[count_df.index.isin(mirna_index + ctrl_index)]
    count_df = count_df.dropna()

In [ ]:
if prefix == "lib2_ActD":
    controls = {
        "P96.2_3UTR": 1,
        "P96.4_3UTR": 1,
        "P96.5_3UTR": 0.1,
        "P96.6_3UTR": 0.1,
        "P96.7_3UTR": 0.01,
        "P96.8_3UTR": 0.01,
        "P96.9_3UTR": 0.001,
        "P96.10_3UTR": 0.001,
        "P96.11_3UTR": 0.0001,
        "P96.14_3UTR": 0.0001,
    }
elif prefix == "lib1_ActD":
    controls = {
        "P88.1_3UTR": 1,
        "P88.3_3UTR": 1,
        "P88.5_3UTR": 0.1,
        "P88.6_3UTR": 0.1,
        "P88.7_3UTR": 0.01,
        "P88.8_3UTR": 0.01,
        "P88.9_3UTR": 0.001,
        "P88.10_3UTR": 0.001,
        "P88.11_3UTR": 0.0001,
        "P88.12_3UTR": 0.0001,
    }

In [ ]:
plt.figure(figsize=(2, 1.6))
for column in count_df.columns:
    r2 = np.corrcoef(np.log10(count_df.loc[controls.keys(), column]), np.log10(list(controls.values())))[0, 1]**2
    plt.scatter(count_df.loc[controls.keys(), column], list(controls.values()), label=f"{column} ($r^2$ = {r2:.3f})", edgecolors="none", s=10)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('sequencing counts')
plt.ylabel('expected abundance')
plt.tight_layout()
plt.legend(ncol=1, fontsize=6)
plt.savefig(os.path.join(plot_folder, f"{prefix}_reference_calibration.svg"))

In [ ]:
def fit_linear_with_intercept(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    a, b = np.polyfit(x, y, 1)
    return a, b

# spike-in IDs (row index labels)
control_ids = list(controls.keys())[::-1]
# true spike-in abundances
true_abund = pd.Series(controls)[::-1]

params = {}
for col in count_df.columns:
    y = count_df.loc[control_ids, col]
    a_j, b_j = fit_linear_with_intercept(true_abund, y)
    params[col] = (a_j, b_j)

# 0h is the reference time point
ref_col = count_df.columns[0]
a_ref, b_ref = params[ref_col]

count_df_adjusted = count_df.copy()
for col in count_df.columns:
    a_j, b_j = params[col]
    count_df_adjusted[col] = count_df[col] * (a_ref / a_j)

# Optional: avoid negative values from extrapolation / noise
count_df_adjusted[count_df_adjusted < 0] = 0

In [ ]:
if prefix == "lib2_ActD":
    count_df_adjusted = count_df_adjusted[count_df_adjusted["0h"] > 20]
    count_df_adjusted = count_df_adjusted[~count_df_adjusted.index.str.contains("P96")]
    
elif prefix == "lib1_ActD":
    count_df_adjusted = count_df_adjusted[count_df_adjusted["0h"] > 30]
    count_df_adjusted = count_df_adjusted[~count_df_adjusted.index.str.contains("P88")]

In [ ]:
def find_relevant_time_points(row):
    row = row/row[0]
    first_small_entry = row[row <= 0.1]
    if len(first_small_entry) > 0:
        last_entry = first_small_entry.index[0]
    else:
        last_entry = "none"
    return last_entry

def void_entries_after_last_entry(row):
    last_entry_col = row['last_entry']
    if last_entry_col == "none":
        return row
    # Find the index of the last_entry column
    last_entry_idx = row.index.get_loc(last_entry_col)
    
    for i in range(last_entry_idx, len(row)):
        if row.index[i] != 'last_entry':  # Don't modify the last_entry column itself
            row.iloc[i] = np.nan
    return row

count_df_adjusted['last_entry'] = count_df_adjusted.apply(find_relevant_time_points, axis=1)
count_df_adjusted = count_df_adjusted.apply(void_entries_after_last_entry, axis=1)
count_df_adjusted = count_df_adjusted.drop(columns=['last_entry'])

In [ ]:
from scipy.optimize import curve_fit

# Define the exponential decay function
def exponential_decay(t, m0, k):
    return m0 * np.exp(-k * t)

examples = {
    "lib1_ActD": ["miRNA_bulged_AND5_14_context1"],
    "lib2_ActD": ["30_miRNA_full_subset_three_target_AND4_16", "34_miRNA_full_all_range_target_log_AND5_89", '8_miRNA_full_repeat_x4_18']
}
plt.figure(figsize=(2, 1.3))
colors = plt.get_cmap('tab10').colors
for i, example in enumerate(examples[prefix]):
    mRNA_levels = count_df_adjusted.loc[example].dropna()
    time_points_curr = time_points[:len(mRNA_levels)]
    # Perform the curve fit
    mRNA_levels = mRNA_levels / mRNA_levels.iloc[0]
    popt, pcov = curve_fit(exponential_decay, time_points_curr, mRNA_levels, p0=(100, 0.1))

    # Extract fitted parameters
    m0_fit, k_fit = popt
    print(f"Fitted Parameters: m0 = {m0_fit:.2f}, k = {k_fit:.4f}")

    # Generate the fitted curve
    time_fit = np.linspace(0, max(time_points_curr), 100)  # Fine-grained time points for plotting
    mRNA_fit = exponential_decay(time_fit, m0_fit, k_fit)

    half_life = np.log(2) / k_fit
    r2 = np.corrcoef(np.log10(mRNA_levels), np.log10(exponential_decay(np.array(time_points_curr), *popt)))[0, 1]**2
    print(r2)
    plt.scatter(time_points_curr, mRNA_levels, color=colors[i], s=20, edgecolors="none")
    plt.plot(time_fit, mRNA_fit, label=r"$t_{1/2}$"+f"={half_life:.1f}h", color=colors[i])
    
plt.xlabel("time (h)")
plt.ylabel(r"log$_{10}$(mRNA Level)")
plt.legend(fontsize=6)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"{prefix}_decay_fit_examples.svg"))
plt.show()    

In [ ]:
total_abundance = count_df_adjusted.sum(axis=0) / count_df_adjusted.sum(axis=0).iloc[0]
plt.figure(figsize=(2, 1.6))
plt.scatter(time_points, total_abundance, marker='o', s=10)

popt, pcov = curve_fit(exponential_decay, time_points, total_abundance.values, p0=(100, 0.1))
half_life_total = np.log(2) / popt[1]

plt.text(7, 0.9, f"overall half-life = {half_life_total:.1f}h", fontsize=7)
plt.ylim(0, 1.05)
plt.xlabel("time (h)")
plt.ylabel("rel. total library abundance")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"{prefix}_total_abundance.svg"))

In [ ]:
half_life_series = pd.Series(index=count_df_adjusted.index, dtype=float)
error_series_m0 = pd.Series(index=count_df_adjusted.index, dtype=float)
error_series_k = pd.Series(index=count_df_adjusted.index, dtype=float)
r2_series = pd.Series(index=count_df_adjusted.index, dtype=float)
for index, row in count_df_adjusted.iterrows():
    mRNA_levels = row.dropna().values
    mRNA_levels_fit = np.array(mRNA_levels)
    time_points_fit = np.array(time_points[:len(mRNA_levels)])
    
    popt, pcov = curve_fit(exponential_decay, time_points_fit, mRNA_levels_fit, p0=(100, 0.1), maxfev=10000)
    m0_fit, k_fit = popt
    half_life = np.log(2) / k_fit
    half_life_series[index] = half_life
    
    error_m0 = np.sqrt(pcov[0,0])
    error_k = np.sqrt(pcov[1,1])
    error_series_m0[index] = error_m0 / m0_fit
    error_series_k[index] = error_k / k_fit
    
    # calculate the r-squared of the fit
    y_fit = exponential_decay(time_points_fit, *popt)
    r2 = np.corrcoef(np.log10(mRNA_levels_fit), np.log10(y_fit))[0, 1]**2
    r2_series[index] = r2

In [ ]:
plt.figure(figsize=(2, 1.6))
# Bin r2 values manually, then take log of counts
bins = np.arange(0, 1.05, 0.025)
binned_r2 = pd.cut(r2_series, bins=bins)
counts = binned_r2.value_counts().sort_index()
plt.bar([interval.mid for interval in counts.index], np.log10(counts.values), width=0.025, color="skyblue", edgecolor="black")
plt.xlabel(f"$r^2$ of exponential decay fit")
plt.ylabel(r"log$_{10}$(count)")
plt.axvline(0.85, color='red', linestyle='dashed', linewidth=1, label="$r^2$ = 0.85 cutoff")
plt.tight_layout()
plt.legend(loc='upper left', fontsize=6)
plt.savefig(os.path.join(plot_folder, f"{prefix}_r2_distribution.svg"))

In [ ]:
def calc_area_under_curve(row):
    row = row/row[0]
    return np.trapz(row.values, time_points)

all_area_under_curve = pd.Series(index=count_df_adjusted.index, dtype=float)
for index, row in count_df_adjusted.iterrows():
    all_area_under_curve[index] = calc_area_under_curve(row)

In [ ]:
half_life_series = half_life_series[~half_life_series.index.isin(controls.keys())]
half_life_series = half_life_series[~half_life_series.index.isin(["P96.orig_3UTR"])]

all_area_under_curve = all_area_under_curve[~all_area_under_curve.index.isin(controls.keys())]
all_area_under_curve = all_area_under_curve[~all_area_under_curve.index.isin(["P96.orig_3UTR"])]

error_series_k = error_series_k[~error_series_k.index.isin(controls.keys())]
error_series_k = error_series_k[~error_series_k.index.isin(["P96.orig_3UTR"])]

error_series_m0 = error_series_m0[~error_series_m0.index.isin(controls.keys())]
error_series_m0 = error_series_m0[~error_series_m0.index.isin(["P96.orig_3UTR"])]

r2_series = r2_series[~r2_series.index.isin(controls.keys())]
r2_series = r2_series[~r2_series.index.isin(["P96.orig_3UTR"])]

In [ ]:
count_df_half_life = count_df_adjusted.copy()
count_df_half_life["half_life"] = half_life_series
count_df_half_life["area_under_curve"] = all_area_under_curve
count_df_half_life["error_k"] = error_series_k
count_df_half_life["error_m0"] = error_series_m0
count_df_half_life["r2"] = r2_series
count_df_half_life["8h_vs_30min"] = count_df_half_life["8h"] / count_df_half_life["05h"]

# filter
count_df_half_life = count_df_half_life[count_df_half_life["r2"] > 0.85]

In [ ]:
if prefix == "lib2_ActD":
    comparison_df = pd.read_csv("plasmid_assay_data/library2_log2fc_with_UMIs_with_K562.csv", index_col=0)
    comparison_df = 2**comparison_df[["K562_3UTR_log2FoldChange"]]
    comparison_df = comparison_df.rename(columns={"K562_3UTR_log2FoldChange": "plasmid"})
elif prefix == "lib1_ActD":
    comparison_df = pd.read_csv("plasmid_assay_data/lib1_log2fc_combined.csv", index_col=0)
    comparison_df = 2**comparison_df[["log2FoldChange_3UTR_K562_DNA"]]
    comparison_df = comparison_df.rename(columns={"log2FoldChange_3UTR_K562_DNA": "plasmid"})
comparison_df = comparison_df.loc[count_df_half_life.index]

In [ ]:
plt.figure(figsize=(2.4, 1.8))
r = np.corrcoef(np.log10(count_df_half_life["half_life"]), np.log10(comparison_df["plasmid"]))[0, 1]**2
plt.scatter(np.log10(count_df_half_life["half_life"]), np.log10(comparison_df["plasmid"]),
    s=3, alpha=1, edgecolors="none", rasterized=True)

plt.text(-0.5, 0.5, f"r$^2$ = {r:.2f}", ha="center", va="center")
plt.xlim(-1, 1.5)
plt.ylim(-2.5, 1)
plt.xlabel(r"log$_{10}$(time-course half-life (h))")
plt.ylabel(r"log$_{10}$(ratiometric stability)")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"{prefix}_half_life_comparison_raw.svg"), dpi=400)

# Combine

In [ ]:
combined_df = pd.concat([count_df_half_life, comparison_df["plasmid"]], axis=1)
combined_df.sort_values(by="plasmid", inplace=True, ascending=False)

In [ ]:
plt.figure(figsize=(2.4, 1.8))
plt.hist(np.log10(combined_df["plasmid"]), bins=np.arange(-2, 0.5, 0.1), alpha=0.5)
binned_stability = pd.cut(np.log10(combined_df["plasmid"]), bins=np.arange(-2, 0.5, 0.1)).value_counts()
mode_stability = binned_stability.idxmax().mid
plt.axvline(mode_stability, color='black', linestyle='--')
plt.xlabel(r"log$_{10}$"+"(raw plasmid stability)")
plt.ylabel("count")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"{prefix}_plasmid_stability_distribution.svg"))

In [ ]:
plt.figure(figsize=(2, 1.6))
binned_half_life = pd.cut(np.log10(combined_df["half_life"]), bins=np.arange(-1, 2, 0.1)).value_counts()
mode_half_life = binned_half_life.idxmax().mid

plt.hist(combined_df["half_life"], bins=np.arange(0, 16, 0.5), alpha=0.5)
plt.axvline(10**mode_half_life, color='black', linestyle='--')
plt.xlabel("half-life (h)")
plt.ylabel("construct count")
plt.tight_layout()
plt.text(10, 800, f"{10**mode_half_life:.1f}h", fontsize=8)
plt.savefig(os.path.join(plot_folder, f"{prefix}_half-life_distribution.svg"))

In [ ]:
combined_df["plasmid_half_life"] = combined_df["plasmid"] * mode_half_life / mode_stability
combined_df = combined_df.sort_values(by="plasmid_half_life", ascending=False)

if not "combined_df_by_prefix" in locals():
    combined_df_by_prefix = {}
combined_df_by_prefix[prefix] = combined_df

In [ ]:
plt.figure(figsize=(2.4, 1.8))
plt.hist(combined_df_by_prefix['lib1_ActD']["plasmid"], bins=np.arange(0, 3, 0.1), alpha=0.5, label="lib1", density=True)
plt.hist(combined_df_by_prefix['lib2_ActD']["plasmid"], bins=np.arange(0, 3, 0.1), alpha=0.5, label="lib2", density=True)
plt.xlabel("raw plasmid stability")
plt.ylabel("count")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"plasmid_stability_distribution.svg"))

# Joint analysis

In [ ]:
plt.figure(figsize=(2.2, 1.8))

r2_lib1 = np.corrcoef(np.log10(combined_df_by_prefix["lib1_ActD"]["half_life"]), np.log10(combined_df_by_prefix["lib1_ActD"]["plasmid_half_life"]))[0, 1]**2
r2_lib2 = np.corrcoef(np.log10(combined_df_by_prefix['lib2_ActD']["half_life"]), np.log10(combined_df_by_prefix['lib2_ActD']["plasmid_half_life"]))[0, 1]**2

linear_fit_lib2 = np.polyfit(np.log10(combined_df_by_prefix['lib2_ActD']["half_life"]), np.log10(combined_df_by_prefix['lib2_ActD']["plasmid_half_life"]), 1)
plt.plot(np.log10(combined_df_by_prefix['lib2_ActD']["half_life"]), np.polyval(linear_fit_lib2, np.log10(combined_df_by_prefix['lib2_ActD']["half_life"])),
         color="black", linewidth=0.5)

linear_fit_lib1 = np.polyfit(np.log10(combined_df_by_prefix['lib1_ActD']["half_life"]), np.log10(combined_df_by_prefix['lib1_ActD']["plasmid_half_life"]), 1)
plt.plot(np.log10(combined_df_by_prefix['lib1_ActD']["half_life"]), np.polyval(linear_fit_lib1, np.log10(combined_df_by_prefix['lib1_ActD']["half_life"])),
         color="black", linewidth=0.5)

plt.scatter(np.log10(combined_df_by_prefix['lib2_ActD']["half_life"]), np.log10(combined_df_by_prefix['lib2_ActD']["plasmid_half_life"]),
    s=1, alpha=0.5, label=f"lib2, a={linear_fit_lib2[0]:.2f}, b={linear_fit_lib2[1]:.2f}", color="tab:blue", edgecolors="none", rasterized=True)
plt.scatter(np.log10(combined_df_by_prefix['lib1_ActD']["half_life"]), np.log10(combined_df_by_prefix['lib1_ActD']["plasmid_half_life"]),
    s=1, alpha=0.5, label=f"lib1, a={linear_fit_lib1[0]:.2f}, b={linear_fit_lib1[1]:.2f}", color="tab:red", edgecolors="none", rasterized=True)

plt.xlim(-1, 1.7)
plt.ylim(-1.7, 2)
plt.xlabel(r"log$_{10}$(ActD-based half-life (h))")
plt.ylabel(r"log$_{10}$(plasmid half-life (h))")
plt.legend(loc="upper left", fontsize=7, markerscale=5)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"half_life_comparison_adjusted.svg"))